# EDA dữ liệu POI Việt Nam

## Load dataset về POI Việt Nam và các đơn vị Hành chính

In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

poi = pq.read_table(ROOT / "data/raw/poi/poi_raw_vn.parquet").to_pandas()
prov = pq.read_table(ROOT / "data/raw/admin/provinces_vnsdi.parquet").to_pandas()

print(f"Number of POIs: {len(poi)}")
print(f"Number of provinces: {len(prov)}")

Number of POIs: 410426
Number of provinces: 34


## Xử lý thô POI

In [4]:
poi.head(10)

,osm_type,osm_id,is_area,lat,lng,area_m2,n_tags,tags,geometry_wkb
0,node,74099711,False,10.710160,105.117498,NaN,16,"{""name"": ""Châu Đốc"", ""name:en"": ""Châu Đốc"", ""n...",None
1,node,74132064,False,21.028333,105.854041,NaN,148,"{""ISO3166-2"": ""VN-HN"", ""admin_level"": ""2"", ""ca...",None
2,node,81799495,False,21.027661,105.841143,NaN,5,"{""crossing:barrier"": ""full"", ""crossing:bell"": ...",None
3,node,82506228,False,21.215142,105.803426,NaN,2,"{""aeroway"": ""gate"", ""ref"": ""9""}",None
4,node,82506242,False,21.214274,105.804203,NaN,2,"{""aeroway"": ""gate"", ""ref"": ""3""}",None
5,node,82565026,False,21.010744,105.841252,NaN,5,"{""crossing:barrier"": ""full"", ""crossing:bell"": ...",None
6,node,82566014,False,20.972217,105.840644,NaN,4,"{""crossing:barrier"": ""full"", ""crossing:bell"": ...",None
7,node,82566621,False,20.974044,105.840543,NaN,2,"{""railway"": ""switch"", ""ref"": ""N12""}",None
8,node,82566623,False,20.975658,105.840391,NaN,2,"{""railway"": ""switch"", ""ref"": ""N18""}",None
9,node,82566625,False,20.975869,105.840333,NaN,2,"{""railway"": ""switch"", ""ref"": ""N24""}",None


Hiện tại 'poi_raw_vn' chỉ có 8 field, do đó để dễ xử lý chúng ta sẽ extend toàn bộ, toạ một bộ dataset trung gian để xử lý

In [5]:
import json
tags = poi["tags"].apply(json.loads)
tags.head(10)

0    {'name': 'Châu Đốc', 'name:en': 'Châu Đốc', 'n...
1    {'ISO3166-2': 'VN-HN', 'admin_level': '2', 'ca...
2    {'crossing:barrier': 'full', 'crossing:bell': ...
3                      {'aeroway': 'gate', 'ref': '9'}
4                      {'aeroway': 'gate', 'ref': '3'}
5    {'crossing:barrier': 'full', 'crossing:bell': ...
6    {'crossing:barrier': 'full', 'crossing:bell': ...
7                  {'railway': 'switch', 'ref': 'N12'}
8                  {'railway': 'switch', 'ref': 'N18'}
9                  {'railway': 'switch', 'ref': 'N24'}
Name: tags, dtype: object

In [6]:
poi['name'] = tags.map(lambda x: x.get('name'))
for key in ["shop", "amenity", "tourism", "leisure", "office", "building",
            "healthcare", "railway", "highway", "power", "landuse", "public_transport"]:
    poi[key] = tags.map(lambda t, k=key: t.get(k))
    
poi["is_boundary"] = tags.map(lambda t: t.get("admin_level") is not None or t.get("boundary") is not None)
poi["area_outlier"] = poi["is_area"] & (poi["area_m2"] > 1_000_000)
poi["dup_coord"] = poi.duplicated(subset=["lat", "lng"], keep=False)

print("is_boundary:", poi["is_boundary"].sum())
print("area_outlier (is_area & area_m2 > 100ha):", poi["area_outlier"].sum())
print("dup_coord:", poi["dup_coord"].sum())

is_boundary: 1146
area_outlier (is_area & area_m2 > 100ha): 3422
dup_coord: 730


In [7]:
from shapely import STRtree, points, wkb

codes = prov["province_code"].to_numpy()
pts = points(poi["lng"].to_numpy(), poi["lat"].to_numpy())

strict_geoms = prov["geometry_wkb"].map(wkb.loads).to_numpy()
tree1 = STRtree(strict_geoms)
inp1, tidx1 = tree1.query(pts, predicate="within")
province_code = np.full(len(poi), None, dtype=object)
province_code[inp1] = codes[tidx1]

unmatched = np.where(province_code == None)[0]
buf_geoms = prov["buffer_5km_wkb"].map(wkb.loads).to_numpy()
tree2 = STRtree(buf_geoms)
inp2, tidx2 = tree2.query(pts[unmatched], predicate="within")
province_code[unmatched[inp2]] = codes[tidx2]

poi["province_code"] = province_code
poi["province_name"] = poi["province_code"].map(prov.set_index("province_code")["province_name"])

print(f"Gán được tỉnh: {poi['province_code'].notna().sum():,} / {len(poi):,}")
print(f"Ngoài mọi đa giác + vành đệm 5km: {poi['province_code'].isna().sum():,}")
# đối chứng: lng > 110° là Trường Sa (Khánh Hòa), không phải lỗi toạ độ
far_east = poi[poi["lng"] > 110]
print("\nĐiểm lng > 110° theo tỉnh:")
print(far_east["province_name"].value_counts(dropna=False))

Gán được tỉnh: 409,354 / 410,426
Ngoài mọi đa giác + vành đệm 5km: 1,072

Điểm lng > 110° theo tỉnh:
province_name
Tỉnh Khánh Hòa    163
Name: count, dtype: int64


In [8]:
import h3

from evcs.core.grid import RES

poi_extended = poi.copy()
poi_extended["h3_r8"] = [h3.latlng_to_cell(a, b, RES) for a, b in zip(poi_extended["lat"], poi_extended["lng"])]

out_path = ROOT / "data/qa/critique/poi_extended_vn.parquet"
poi_extended.to_parquet(out_path, index=False)

print(f"poi_extended: {len(poi_extended):,} dòng, {poi_extended.shape[1]} cột → {out_path}")
print(poi_extended.dtypes)

poi_extended: 410,426 dòng, 28 cột → /home/n91ym1nhky/Work/internVSF/evcs-atlas/data/qa/critique/poi_extended_vn.parquet
osm_type                str
osm_id                int64
is_area                bool
lat                 float64
lng                 float64
area_m2             float64
n_tags                int32
tags                    str
geometry_wkb         object
name                    str
shop                    str
amenity                 str
tourism                 str
leisure                 str
office                  str
building                str
healthcare              str
railway                 str
highway                 str
power                   str
landuse                 str
public_transport        str
is_boundary            bool
area_outlier           bool
dup_coord              bool
province_code           str
province_name           str
h3_r8                   str
dtype: object


Để dễ phân tích và tập trung vào các tỉnh trọng điểm thì tôi sẽ phân tích các tình thành lớn có cở sở hạ tầng lớn và có nhiều xe điện: Hà Nội, Hồ Chí Minh, Đà Nẵng, Cần Thơ, Đồng Nai, Quảng Ninh, Hải Phòng.
và EDA các POI ở đây để lọc chuẩn, và mong rằng các tỉnh khác cũng sẽ có insight (nghĩa là phuoươ án lọc nhiễu được đề xuất ở các tỉnh trên cuũn có thể áp dụng) tương tự

In [9]:
TARGET = {"01": "Hà Nội", "79": "TP.HCM", "48": "Đà Nẵng", "92": "Cần Thơ",
          "75": "Đồng Nai", "22": "Quảng Ninh", "31": "Hải Phòng"}

poi_7tinh = poi_extended[poi_extended["province_code"].isin(TARGET)].copy()

out_path = ROOT / "data/qa/critique/poi_extended_7tinh.parquet"
poi_7tinh.to_parquet(out_path, index=False)

print(f"poi_7tinh: {len(poi_7tinh):,} dòng / {len(poi_extended):,} dòng toàn quốc → {out_path}")
print()
print(poi_7tinh["province_name"].value_counts())

poi_7tinh: 193,509 dòng / 410,426 dòng toàn quốc → /home/n91ym1nhky/Work/internVSF/evcs-atlas/data/qa/critique/poi_extended_7tinh.parquet

province_name
Thành phố Hồ Chí Minh    61404
Thành phố Hà Nội         57894
Thành phố Đà Nẵng        22969
Tỉnh Đồng Nai            19527
Thành phố Hải Phòng      12288
Thành phố Cần Thơ        11707
Tỉnh Quảng Ninh           7720
Name: count, dtype: int64


In [10]:
print(poi_7tinh.head(10))

   osm_type    osm_id  is_area        lat         lng  area_m2  n_tags  \
1      node  74132064    False  21.028333  105.854041      NaN     148   
2      node  81799495    False  21.027661  105.841143      NaN       5   
3      node  82506228    False  21.215142  105.803426      NaN       2   
4      node  82506242    False  21.214274  105.804203      NaN       2   
5      node  82565026    False  21.010744  105.841252      NaN       5   
6      node  82566014    False  20.972217  105.840644      NaN       4   
7      node  82566621    False  20.974044  105.840543      NaN       2   
8      node  82566623    False  20.975658  105.840391      NaN       2   
9      node  82566625    False  20.975869  105.840333      NaN       2   
10     node  82566626    False  20.976051  105.840355      NaN       2   

                                                 tags geometry_wkb    name  \
1   {"ISO3166-2": "VN-HN", "admin_level": "2", "ca...         None  Hà Nội   
2   {"crossing:barrier": "ful

## Phân tích các loại POI

Theo như yêu cần nhận được là các địa điểm cần đuượ deleviry là:
- Chung cư
- Trung tâm thương mại
- công cộng, khu vui chơi
- bệnh viện, trường học


nhưng để phân tích rõ hơn về hành vi người dùng(nhằm phục vụ tốt nhất) và hướng phần loại các POI sâu hơn thì tôi chia POI vào các nhóm lớn như sau:
- transit: cây xăng, bến xe, bãi đỗ ô tô, sân bay, ga, cảng
- nơi ở nhiều người: chung cư(chung cư, nhà ở xã hội, ..), khách sạn, nhà nghỉ
- thương mại: trung tâm thương mại, siêu thị(loại lớn), showroom
- các địa điểm giải trí: rạp phim, sân vận động, công viên, khu vui chơi, cà phê, nhà hàng, ...
- du lịch: các điểm tham quan(bảo tàng, đền chùa, bảo tàng, bải biển,...)
- công sở, dịch vụ công: cơ quan hành chính, toà văn phòng, ngân hàng, bưu điện
- y tế, giáo dục: bệnh viện, phòng khám lớn, các trường học
- khu công nghiệp: nhà máy, kho bãi, 

để delivery nhanh nhất thì tạm thời xử lý các class sau:
- Chung cư, khu đô thị, 
- khách sạn(loại lớn), resort
- TTTM, siêu thị, showroom
- Công viên, SVĐ, Khu vui chơi, 
- điểm tham quan, bảo tàng, rạp phim
- Trường học
- Bệnh viện
- Văn phòng
- Cơ quan hành chính
- Các mục còn lại sẽ được phân tích sau